# Predicting NYC Housing Prices [Work in Progress]
# Part 1. Data Cleaning & Exploratory Analysis

## Goal
Prepare raw data for analysis by handling missing values, outliers, and inconsistencies, then explore key patterns.

## Inputs & Outputs
- **Input**: `data/raw/housing.csv`
- **Output**: `data/processed/housing_cleaned.csv`

## Key Steps
1. **Data Quality Assessment**: Check for duplicates, missing values, and data types.
2. **Cleaning**: 
   - Remove price outliers (e.g., integer overflow errors)
   - Handle missing values (impute `propertysqft`, `bath`)
3. **Exploratory Data Analysis (EDA)**:
   - Distribution analysis of price and key features
   - Correlation analysis
   - Segment analysis (Luxury vs. Market Rate)


# Load, clean and first look the data

In [133]:
# Import necessary libraries
import pandas as pd
import numpy as np

In [134]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import probplot

In [135]:
import geopandas as gpd

In [136]:
from sklearn.utils.parallel import Parallel, delayed

In [137]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.utils.parallel")

In [138]:
# Import dataset
file = '../data/raw/NY-House-Dataset.csv'

In [139]:
# load the data
df = pd.read_csv(file)

In [140]:
# Inspect the data
df.head()

,BROKERTITLE,TYPE,PRICE,BEDS,BATH,PROPERTYSQFT,ADDRESS,STATE,MAIN_ADDRESS,ADMINISTRATIVE_AREA_LEVEL_2,LOCALITY,SUBLOCALITY,STREET_NAME,LONG_NAME,FORMATTED_ADDRESS,LATITUDE,LONGITUDE
0,Brokered by Douglas Elliman -111 Fifth Ave,Condo for sale,315000,2,2.000000,1400.0,2 E 55th St Unit 803,"New York, NY 10022","2 E 55th St Unit 803New York, NY 10022",New York County,New York,Manhattan,East 55th Street,Regis Residence,"Regis Residence, 2 E 55th St #803, New York, N...",40.761255,-73.974483
1,Brokered by Serhant,Condo for sale,195000000,7,10.000000,17545.0,Central Park Tower Penthouse-217 W 57th New Yo...,"New York, NY 10019",Central Park Tower Penthouse-217 W 57th New Yo...,United States,New York,New York County,New York,West 57th Street,"217 W 57th St, New York, NY 10019, USA",40.766393,-73.980991
2,Brokered by Sowae Corp,House for sale,260000,4,2.000000,2015.0,620 Sinclair Ave,"Staten Island, NY 10312","620 Sinclair AveStaten Island, NY 10312",United States,New York,Richmond County,Staten Island,Sinclair Avenue,"620 Sinclair Ave, Staten Island, NY 10312, USA",40.541805,-74.196109
3,Brokered by COMPASS,Condo for sale,69000,3,1.000000,445.0,2 E 55th St Unit 908W33,"Manhattan, NY 10022","2 E 55th St Unit 908W33Manhattan, NY 10022",United States,New York,New York County,New York,East 55th Street,"2 E 55th St, New York, NY 10022, USA",40.761398,-73.974613
4,Brokered by Sotheby's International Realty - E...,Townhouse for sale,55000000,7,2.373861,14175.0,5 E 64th St,"New York, NY 10065","5 E 64th StNew York, NY 10065",United States,New York,New York County,New York,East 64th Street,"5 E 64th St, New York, NY 10065, USA",40.767224,-73.969856


In [141]:
# Check the dataset's structure
print(f"Number of rows: {df.shape[0]}, Number of columns: {df.shape[1]}")
print("\nColumn names and data types:")
print(df.info())

Number of rows: 4801, Number of columns: 17

Column names and data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4801 entries, 0 to 4800
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   BROKERTITLE                  4801 non-null   object 
 1   TYPE                         4801 non-null   object 
 2   PRICE                        4801 non-null   int64  
 3   BEDS                         4801 non-null   int64  
 4   BATH                         4801 non-null   float64
 5   PROPERTYSQFT                 4801 non-null   float64
 6   ADDRESS                      4801 non-null   object 
 7   STATE                        4801 non-null   object 
 8   MAIN_ADDRESS                 4801 non-null   object 
 9   ADMINISTRATIVE_AREA_LEVEL_2  4801 non-null   object 
 10  LOCALITY                     4801 non-null   object 
 11  SUBLOCALITY                  4801 non-null   object 
 12  ST

First Look Observations:
1. BROKERTITLE - Appears to just contain broker names (likely not useful for analysis)
2. BEDS (int) vs BATHS (float) - different data types
3. Address fields - Multiple address columns with potential redundancy:
   - ADDRESS, MAIN_ADDRESS, FORMATTED_ADDRESS all seem similar.
4. STATE - actually contains ZIP codes
5. Administrative geography fields - Inconsistent granularity:
   - ADMINISTRATIVE_AREA_LEVEL_2 mixes counties with country
   - STREET_NAME contains both street names and area names
   - SUBLOCALITY seems most promising for neighborhood analysis


- Consider creating 'NEIGHBORHOOD' and 'BOROUGH'/'COUNTY' feature
- Address fields could be helpfull if geocoding will be needed

In [142]:
# Inspect numerative features
df.describe()

,PRICE,BEDS,BATH,PROPERTYSQFT,LATITUDE,LONGITUDE
count,4.801000e+03,4801.000000,4801.000000,4801.000000,4801.000000,4801.000000
mean,2.356940e+06,3.356801,2.373861,2184.207862,40.714227,-73.941601
std,3.135525e+07,2.602315,1.946962,2377.140894,0.087676,0.101082
min,2.494000e+03,1.000000,0.000000,230.000000,40.499546,-74.253033
25%,4.990000e+05,2.000000,1.000000,1200.000000,40.639375,-73.987143
50%,8.250000e+05,3.000000,2.000000,2184.207862,40.726749,-73.949189
75%,1.495000e+06,4.000000,3.000000,2184.207862,40.771923,-73.870638
max,2.147484e+09,50.000000,50.000000,65535.000000,40.912729,-73.702450


Right-skewness evident from:

Price: Median 2,147,484,000<br>
Beds: Median 3 vs Max 50<br>
Bath: Median 2 vs Max 50<br>
Sqft: Median 2184.2 vs Max 65535.0<br>

Option 1: Apply log transformation<br>
Option 2: Analyze luxury segment separately<br>
Option 3: Winsorize extreme values<br>
<br>
Suspicious pattern detected in PROPERTYSQFT:

Exact same value (2184.21) appears as:
Mean, 50th percentile (median), 75th percentile<br>
This suggests: Missing values may have been imputed with this number

In [143]:
# Check for Nulls
df.isna().sum()

BROKERTITLE                    0
TYPE                           0
PRICE                          0
BEDS                           0
BATH                           0
PROPERTYSQFT                   0
ADDRESS                        0
STATE                          0
MAIN_ADDRESS                   0
ADMINISTRATIVE_AREA_LEVEL_2    0
LOCALITY                       0
SUBLOCALITY                    0
STREET_NAME                    0
LONG_NAME                      0
FORMATTED_ADDRESS              0
LATITUDE                       0
LONGITUDE                      0
dtype: int64

No null data

In [ ]:
# Check for duplicates
df.duplicated().sum()

In [ ]:
# Drop duplicates
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
# Reset index after cleaning
df = df.reset_index(drop=True)

In [ ]:
# Standartize column names
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [ ]:
df.columns

In [ ]:
df.bath = df.bath.astype(int)

In [ ]:
original_length = df.shape[0]

In [ ]:
original_length

## Exploratory Data Analysis (EDA)

In [ ]:
# Frist create list of features that will be used for EDA only
eda_features = []

## Price distribution analysis

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x=df.price, color='skyblue')

# Add informative titles and labels
plt.title('New York Housing Price Distribution', fontsize=14)
plt.xlabel('Price ($)', fontsize=12)
plt.tight_layout()

# Display plot
plt.show()

In [ ]:
# Right-skewed price distribution. Should check for error data

In [ ]:
df.sort_values(by=['price'], ascending=False).head()

In [ ]:
# 2147483647 is error input, should be dropped

In [ ]:
df = df[df.price < df.price.max()]

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x=df.price, color='skyblue')

# Add informative titles and labels
plt.title('New York Housing Price Distribution', fontsize=14)
plt.xlabel('Price ($)', fontsize=12)
plt.tight_layout()

# Display plot
plt.show()

In [ ]:
# Still have an outlier on 195000000$.I checked this property, apparently it is a very famous ultra expensive apartment.
# I will also exclude it

In [ ]:
df = df[df.price < df.price.max()]

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x=df.price, color='skyblue')

# Add informative titles and labels
plt.title('New York Housing Price Distribution', fontsize=14)
plt.xlabel('Price ($)', fontsize=12)
plt.tight_layout()

# Display plot
plt.show()

In [ ]:
# Now boxplot looks reasonable

In [ ]:
# Apply natural log transformation to price
df['log_price'] = np.log(df.price)

In [ ]:
# Identify outliers on Q-Q plot
probplot(df['log_price'], dist="norm", plot=plt)
plt.title("Q-Q Plot of log prices")
plt.show()

In [ ]:
# I see three outliers in lower range

In [ ]:
df.sort_values('price').head(15)

In [ ]:
# Data cleaning:
# - $2,494 and $3,225 (Manhattan apartments) - Below realistic thresholds
# - $5,800 (land) - Far below NYC market minimums

df = df[df.price>10000]

In [ ]:
# Reset index after cleaning
df = df.reset_index(drop=True)

In [ ]:
# Create price distribution visualization by property type
plt.figure(figsize=(12, 8))
sns.boxplot(df, x='price', y='type', hue='type')

In [ ]:
# Here we can see that we have a stratification of the market, apparently townhouses are considered more as luxurious property

In [ ]:
# Calculate 90th percentile price thresholds by property type to identify price ceilings 
# for the top 10% of each property type and understand price distribution characteristics across categories

type_thresholds = df.groupby('type')['price'].quantile(0.9) 

In [ ]:
print('0.90 quantile by type of houses')
print(type_thresholds.sort_values())
print('mean:',type_thresholds.mean())
print('median:',type_thresholds.median())

In [ ]:
# We again observe that the townhouse category is significantly different

In [ ]:
# Calculate 50th percentile price thresholds by property type to assess entry - middle division point

type_thresholds_50 = df.groupby('type')['price'].quantile(0.5)
print('0.50 quantile by type of houses')
print(type_thresholds_50.sort_values())
print('mean:',type_thresholds_50.mean())
print('median:',type_thresholds_50.median())

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(df, x='log_price', y='type', hue='type')

In [ ]:
sns.histplot(df.log_price, bins=20, kde=True)

In [ ]:
# log_price looks quite normal

In [ ]:
# I see three outliers in lower range

In [ ]:
df.sort_values('price').head()

In [ ]:
# Data cleaning:
# - $2,494 and $3,225 (Manhattan apartments) - Below realistic thresholds
# - $5,800 (land) - Far below NYC market minimums

df = df[df.price>10000]

In [ ]:
# Reset index after cleaning
df = df.reset_index(drop=True)

In [ ]:
1 - len(df)/original_length

In [ ]:
# dropped only 0.1% of data

## Other variables distribution

In [ ]:
df.columns

### _Brokertitle_

In [ ]:
df['broker_count'] = df['brokertitle'].map(df['brokertitle'].value_counts())

In [ ]:
LUXURY_THRESHOLD = 2_000_000 

In [ ]:
# Get broker stats: count and median price
broker_stats = df.groupby('brokertitle').agg(
    num_listings=('price', 'count'),
    median_price=('price', 'median'),
    max_price=('price', 'max')
).sort_values('median_price', ascending=False)

# Filter brokers with ≥5 listings (avoid noise)
broker_stats = broker_stats[broker_stats['num_listings'] >= 5]

# Plot
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=broker_stats,
    x='num_listings',
    y='median_price',
    hue='median_price',
    palette='viridis',
    size='num_listings',
    sizes=(20, 200)
)

# Annotate top luxury brokers
top_luxury = broker_stats[broker_stats['median_price'] > LUXURY_THRESHOLD ]  # Adjust thresholdb
for idx, row in top_luxury.iterrows():
    plt.annotate(
        idx.replace('Brokered by ', '').split('-')[0].strip(),
        (row['num_listings'], row['median_price']),
        fontsize=8, alpha=0.75
    )

#plt.xscale('log')  # Better visibility for wide-ranging counts
plt.xlabel('Number of Listings')
plt.ylabel('Median Price ($)')
plt.title('Broker Price Tier vs. Volume')
plt.axhline(y=3_000_000, color='red', linestyle='--', label='Luxury threshold') 
plt.legend()
plt.show()

In [ ]:
# Plot
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=broker_stats,
    x='num_listings',
    y='max_price',
    hue='max_price',
    palette='viridis',
    size='num_listings',
    sizes=(20, 200)
)

# Annotate top luxury brokers
top_luxury = broker_stats[broker_stats['max_price'] > 20_000_000]  # Adjust thresholdb
for idx, row in top_luxury.iterrows():
    plt.annotate(
        idx.replace('Brokered by ', '').split('-')[0].strip(),
        (row['num_listings'], row['max_price']),
        fontsize=8, alpha=0.75
    )

#plt.xscale('log')  # Better visibility for wide-ranging counts
plt.xlabel('Number of Listings')
plt.ylabel('Max Price ($)')
plt.title('Broker Price Tier vs. Volume')
plt.axhline(y=20_000_000, color='red', linestyle='--', label='Luxury threshold') 
plt.legend()
plt.show()

In [ ]:
broker_stats.head(10)

In [ ]:
broker_stats.sort_values(by='num_listings', ascending=False).head(10)

In [ ]:
df.sort_values(by='price', ascending=False).head(15)

In [ ]:
selected_brokers = [
    'COMPASS',
    'Sotheby\'s International Realty - East Side Manhattan Brokerage',
    'Peter Ashe Real Estate- Peter Ashe Real Estate',
    'Douglas Elliman - 575 Madison Ave',
    'Corcoran East Side',
    'Serhant',
    'Garfield, Leslie J. & Co., Inc.',
    "CHRISTIE'S INT.REAL ESTATE GROUP",
    'Coldwell Banker Warburg',
]

# Filter data
plot_data = df[df['brokertitle'].str.contains('|'.join(selected_brokers))]

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(
    data=plot_data,
    x='brokertitle',
    y='price',
    order=plot_data.groupby('brokertitle')['price'].median().sort_values(ascending=False).index,
    showfliers=True  # Hide outliers initially
)

# Add context
plt.axhline(y=3_000_000, color='red', linestyle='--', label='Luxury threshold')
plt.xticks(rotation=45, ha='right')
#plt.yscale('log')  # Critical for NYC price ranges
plt.title('Price Distribution by Broker')
plt.ylabel('Price')
plt.xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
broker_stats

In [ ]:
LUXURY_THRESHOLD = 2_000_000 

# Calculate Luxury Score for each broker
luxury_scores = (
    df[(df['price'] >= LUXURY_THRESHOLD) & (df['broker_count']>5)]  # Filter luxury properties
    .groupby('brokertitle')              # Group by broker
    .size() /                            # Count luxury listings per broker
    df.groupby('brokertitle').size()     # Divide by broker's total listings
).fillna(0)                              # Handle brokers with no luxury properties

# Add scores back to original DataFrame
df['eda_luxury_score'] = df['brokertitle'].map(luxury_scores)

In [ ]:
df.groupby('brokertitle')[['eda_luxury_score', 'price']].median().sort_values('eda_luxury_score', ascending=False).head(10)

In [ ]:
df['eda_broker_count'] = df['brokertitle'].map(df['brokertitle'].value_counts())

In [ ]:
break

In [ ]:
# Add new features based on brokertitle column (See Feature Engineering section)

### _Type_

In [ ]:
df.type.value_counts()

In [ ]:
plt.figure(figsize=(12, 8))
sns.countplot(data=df, y='type', hue='type')

In [ ]:
# Identify types to potentially consolidate
rare_types = df['type'].value_counts()[df['type'].value_counts() < 20].index
print(f"Rare types (<20 listings): {list(rare_types)}")

In [ ]:
# Group rare types in other category
eda_features.append('eda_type_clean')
df['eda_type_clean'] = np.where(df['type'].isin(rare_types), 'Other', df['type'])

In [ ]:
# Averages by type
df.groupby('eda_type_clean')[['price', 'beds', 'bath', 'propertysqft', ]].mean().sort_values('price', ascending=False)

In [ ]:
# Here we also can see that townhouse type differs from others

In [ ]:
# Calculate market share by types
market_share = (df.groupby('eda_type_clean')['price'].sum() / df.price.sum() * 100).sort_values(ascending=False)
plt.figure(figsize=(12, 6))
market_share_df = market_share.reset_index()
sns.barplot(data=market_share_df,
            x='price',
            y='eda_type_clean',
            hue='eda_type_clean')

In [ ]:
market_share

In [ ]:
# Additionally consolidate rare property types (<1% market share) into 'Other' category
df['eda_type_clean'] = np.where(df['eda_type_clean'].isin(market_share[market_share < 1.0].index),'Other', df['eda_type_clean'])

In [ ]:
# Calculate market share by types
market_share = (df.groupby('eda_type_clean')['price'].sum() / df.price.sum() * 100).sort_values(ascending=False)
plt.figure(figsize=(12, 6))
market_share_df = market_share.reset_index()
sns.barplot(data=market_share_df,
            x='price',
            y='eda_type_clean',
            hue='eda_type_clean')

In [ ]:
# Create price distribution visualization by property type
plt.figure(figsize=(12, 8))
sns.boxplot(df, x='price', y='eda_type_clean', hue='eda_type_clean')

### _Beds_

In [ ]:
plt.figure(figsize=(12, 8))
sns.countplot(data=df, y='beds', hue='beds')

In [ ]:
# Here we also have a skewed distribution

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df,  # Filter outliers
    x='beds',
    y='price',
    showfliers=True,
    hue='beds'
)
#plt.yscale('log')
plt.title('Price Distribution by Bedroom Count', fontweight='bold')

In [ ]:
# Market doesn't following beds after 10-12 bedrooms. Most high priced properties have 1-8 bedrooms. So it's probably a good idea divide those groups

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df,  # Filter outliers
    x='beds',
    y='price',
    showfliers=False,
    hue='beds'
)
#plt.yscale('log')
plt.title('Price Distribution by Bedroom Count', fontweight='bold')

In [ ]:
# Here we also see linear relation beds - price up to 10 bedrooms

In [ ]:
# Those properties with high bedroom number are mostly multi-family homes

In [ ]:
# ok, so no luxurios entries in high number bedroom segment

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=df, x='beds', hue='beds')

In [ ]:
df['beds'].quantile(0.99)

In [ ]:
# So 12 bedrooms seems as a good threshold. Feature will be added in Feature Engineering section

In [ ]:
# Calculate market share by bedrooms
market_share = (df.groupby('beds')['price'].sum() / df.price.sum() * 100).sort_values(ascending=False)
plt.figure(figsize=(12, 6))
market_share_df = market_share.reset_index()
sns.barplot(data=market_share_df,
            y='price',
            x='beds',
            hue='beds',
            palette='Blues_d')

In [ ]:
# Mean price dinamic based on number of bedrooms
market_share = (df.groupby('beds')['price'].mean()).sort_values(ascending=False)
plt.figure(figsize=(12, 6))
market_share_df = market_share.reset_index()
sns.barplot(data=market_share_df,
            y='price',
            x='beds',
            hue='beds',
            palette='Blues_d')

In [ ]:
# The 1-6 bedroom homes aren’t the most expensive, but they dominate the market – they make up nearly 90% of listings.

In [ ]:
df['log_beds'] = np.log1p(df['beds'])

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(df.log_beds)

In [ ]:
df['beds_sqrt'] = np.sqrt(df['beds'])

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(df.beds_sqrt)

In [ ]:
df['beds_asinh'] = np.arcsinh(df['beds'])

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(df.beds_asinh)

In [ ]:
bins = [0, 1, 2, 3, 4, 5, 6, 10, 20, 50]
labels = ['1', '2', '3', '4', '5', '6', '7-10', '11-20', '20+']
df['beds_binned'] = pd.cut(df['beds'], bins=bins, labels=labels)

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(df.beds_binned)

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='beds_binned', y='price', data=df, order=labels, hue='eda_type_clean')

### Bath

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=df, y='bath', hue='bath')

In [ ]:
df.bath.describe()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df,  # Filter outliers
    x='bath',
    y='price',
    showfliers=True,
    hue='bath'
)
#plt.yscale('log')
plt.title('Price Distribution by Bathroom Count', fontweight='bold')

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(df,y='bath', hue='type')

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df, x='bath', y='log_price', hue='eda_type_clean')

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='bath', y='price', hue='eda_type_clean', showfliers=False)

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='bath', y='log_price', hue='eda_type_clean', showfliers=False)

In [ ]:
# Mean price dinamic based on number of bathrooms
market_share = (df.groupby('bath')['price'].mean()).sort_values(ascending=False)
plt.figure(figsize=(12, 6))
market_share_df = market_share.reset_index()
sns.barplot(data=market_share_df,
            y='price',
            x='bath',
            hue='bath',
            palette='Blues_d')

In [ ]:
df['log_bath'] = np.log1p(df.bath)

In [ ]:
plt.figure(figsize=(12, 8))
sns.histplot(df.log_bath)

In [ ]:
bins = [0, 1, 2, 3, 4, 6, 10, 50]
labels = ['1', '2', '3', '4', '5-6', '7-10', '10+']
df['bath_binned'] = pd.cut(df['bath'], bins=bins, labels=labels, include_lowest=True)

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=df, y='bath_binned')

### *Propertysqft*

In [ ]:
sns.histplot(df.propertysqft)

In [ ]:
df.propertysqft.value_counts()

In [ ]:
df.propertysqft.describe()

In [ ]:
# The value 2184.207862 appears to have been used as a substitute for missing values.

In [ ]:
df[df.propertysqft!=2184.207862].propertysqft.describe()

In [ ]:
# On the dataset without 2184.207862, the median is 1495.50. I’ll impute missing values with this median and mark them with the area_missing flag.

In [ ]:
df['area_missing'] = np.where(df['propertysqft'] == 2184.207862, 1, 0)

In [ ]:
df['area_missing'].value_counts()

In [ ]:
df['propertysqft'] = np.where(df['area_missing'] == 1, 1495.512345, df['propertysqft'])

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(data=df, y='eda_type_clean', x='price', hue='area_missing')

In [ ]:
# Do not see here significal disproportions

In [ ]:
df.propertysqft.describe()

In [ ]:
sns.boxplot(df.propertysqft)

In [ ]:
sns.histplot(df.propertysqft, bins=10)

In [ ]:
df['propertysqft'].quantile(0.995)

In [ ]:
# Propertysqft data looks highly skewed. I'll apply log transform to compensate

In [ ]:
df['log_area'] = np.log1p(df.propertysqft)

In [ ]:
sns.histplot(df.log_area, bins=10)

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df, x='log_area', y='log_price', hue='eda_type_clean')

In [ ]:
# In this graph, we can observe data clustering. The vertical line of dots corresponds to the substituted value that was used 
# to fill in missing entries. The most expensive properties across all area ranges are condos.

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(df[df.area_missing==True], y='eda_type_clean', hue='eda_type_clean')

In [ ]:
# Mean price dinamic based on type
market_share = (df.groupby('eda_type_clean')['price'].sum()).sort_values(ascending=False)
plt.figure(figsize=(12, 6))
market_share_df = market_share.reset_index()
sns.barplot(data=market_share_df,
            y='price',
            x='eda_type_clean',
            hue='eda_type_clean')

In [ ]:
# Mean price dinamic based on type
market_share = (df.groupby('eda_type_clean')['price'].mean()).sort_values(ascending=False)
plt.figure(figsize=(12, 6))
market_share_df = market_share.reset_index()
sns.barplot(data=market_share_df,
            y='price',
            x='eda_type_clean',
            hue='eda_type_clean')

In [ ]:
# Most properites with missed values are co-op, and that item has the least mean price. So those missing properties shouldn't
# distort upper market.

### All address data

In [ ]:
address_columns = ['address', 'state',
       'main_address', 'administrative_area_level_2', 'locality',
       'sublocality', 'street_name', 'long_name', 'formatted_address']

In [ ]:
df[address_columns].head(7)

In [ ]:
# Address and formatted address probably should be dropped.

In [ ]:
# State contain zip. Could be usefull

In [ ]:
df['zip'] = df['state'].str[-5:]

# Filter invalid ZIPs (e.g., non-numeric)
df['zip'] = df['zip'].where(df['zip'].str.isdigit())

In [ ]:
df.zip.isna().sum()

In [ ]:
df.zip.value_counts()

In [ ]:
df.state.value_counts()

In [ ]:
df.administrative_area_level_2.value_counts()

In [ ]:
# Messy feature, better should use borough

In [ ]:
#locality	sublocality	street_name	long_name

In [ ]:
df.locality.value_counts()

In [ ]:
df.sublocality.value_counts()

In [ ]:
df.street_name.value_counts()

In [ ]:
df.long_name.value_counts()

In [ ]:
address_columns_to_drop = ['address',
 'state',
 'main_address',
 'administrative_area_level_2',
 'locality',
 'sublocality',
 'street_name',
 'long_name']

In [ ]:
df = df.drop(columns=address_columns_to_drop)

In [ ]:
# Reset index after cleaning
df = df.reset_index(drop=True)

In [ ]:
df.columns

### Add boroughs

In [ ]:
boroughs_file = '../data/raw/nyc_boroughs.geojson'

In [ ]:
gdf_boroughs = gpd.read_file(boroughs_file)

In [ ]:
gdf_boroughs.plot()

In [ ]:
gdf_boroughs

In [ ]:
gdf_boroughs_utm = gdf_boroughs.to_crs("EPSG:32618")

In [ ]:
gdf_houses = gpd.GeoDataFrame(
    df, 
    geometry=gpd.points_from_xy(df['longitude'], df['latitude']),
    crs=gdf_boroughs.crs
)

In [ ]:
df_with_borough = gpd.sjoin(
    gdf_houses,
    gdf_boroughs[['name', 'geometry']],  # Only keep needed columns
    how='left',
    predicate='within'
)

In [ ]:
df_with_borough

In [ ]:
df['borough'] = df_with_borough['name']

In [ ]:
df['borough'].isna().sum()

In [ ]:
# Nothing missing

In [ ]:
plt.figure(figsize=(12, 8))
sns.scatterplot(x="longitude", y="latitude", hue="borough", data=df)

In [ ]:
# Looks great!

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(df, x='price', y='borough', hue='borough')

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(df, x='log_price', y='borough', hue='borough')

In [ ]:
df.columns

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(df, x='beds', y='borough', hue='borough')

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='beds_binned', y='price', data=df, order=labels, hue='borough')

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(df, x='bath', y='borough', hue='borough')

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='borough', y='price', hue='eda_type_clean', showfliers=False)

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=df, y='borough', hue='borough')

In [ ]:
# Calculate market share by types
market_share = (df.groupby('borough')['price'].sum() / df.price.sum() * 100).sort_values(ascending=False)
plt.figure(figsize=(12, 6))
market_share_df = market_share.reset_index()
sns.barplot(data=market_share_df,
            x='price',
            y='borough',
           hue='borough')

In [ ]:
# Ok, those plots confirm what we already know, all the money is in Manhattan.

In [ ]:
df.borough.value_counts()

In [ ]:
# Enconding for boroughs. Only top boroughs
borough_dummies = pd.get_dummies(df['borough'], prefix='borough')

top_boroughs = ['borough_Manhattan', 'borough_Brooklyn', 'borough_Queens']
df = pd.concat([df, borough_dummies[top_boroughs]], axis=1)

In [ ]:
df.columns

In [ ]:
print(np.log(60_000_000))
print(np.log(30_000_000))
print(np.log(10_000_000))
print(np.log(2_000_000))
print(np.log(1_000_000))
print(np.log(500_000))

In [ ]:
### Add price_per_sqft

In [ ]:
df['price_per_sqft'] = df.price/df.propertysqft

In [ ]:
df['log_pps'] = np.log1p(df.price_per_sqft)

In [ ]:
sns.histplot(df, x='price_per_sqft')

In [ ]:
plt.figure(figsize=(12,8))
sns.boxplot(df, x='price_per_sqft', y='eda_type_clean', hue='eda_type_clean')

In [ ]:
df[['price_per_sqft', 'price']].corr()

In [ ]:
plt.figure(figsize=(12,8))
sns.boxplot(df, x='price_per_sqft', y='borough', hue='borough')

In [ ]:
plt.figure(figsize=(12,8))
sns.boxplot(df, x='price_per_sqft', y='borough', hue='eda_type_clean')

## Correlation

In [ ]:
df.columns

In [ ]:
numerical_columns = ['price', 'beds', 'bath', 'propertysqft',
       'log_price','log_beds', 'log_bath',
       'log_area','price_per_sqft']

In [ ]:
corr = df[numerical_columns].corr()

In [ ]:
sns.heatmap(corr)

In [ ]:
sns.pairplot(df[numerical_columns])

## Save edited dataset

In [ ]:
df.to_csv('../data/processed/housing_cleaned.csv', index=False)